# 00. ELT 파이프라인 — 코스메틱 이커머스 이벤트 로그

## 분석 개요

- 분석 목적: 코스메틱 이커머스 원본 이벤트 로그 5개월치(약 2,069만 행 / 2.3GB)를 MySQL `events` 테이블 하나로 적재해 이후 모든 분석의 단일 소스로 만든다.
- 분석 단위: 이벤트(원본 1행 = 1이벤트). 세션·사용자 파생은 이후 노트북에서 수행한다.
- 적재 원칙: 이 단계는 필터·중복 제거·순서 판정을 하지 않고 원본을 그대로 싣는다. 결측·가격 이상값의 규모는 적재 후 3. 검증에서 확인하고, 품질 진단·제외 기준은 `01_eda`에서 확정한다.

## 분석 흐름
1. **데이터 개요** — 파일 구조·공유 스키마 파악
2. **적재** — `events` 테이블 생성 → 월별·청크 적재 → 인덱스 → MySQL 저장
3. **검증** — 원본 사전 스캔값과 대조

In [1]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

load_dotenv()  # 프로젝트 루트의 .env 자동 탐색
DATA_DIR = os.getenv('DATA_DIR')

---
## 1. 데이터 개요

- **데이터 출처**: https://www.kaggle.com/datasets/mkechinov/ecommerce-events-history-in-cosmetics-shop
- **분석 기간**: 2019-10-01 ~ 2020-02-29 (연속 5개월)
- 월별로 분리된 5개 CSV이며 **스키마가 동일**하다. 하나의 이벤트 로그를 월 단위로 나눈 구조다.

| 파일 | 기간 |
|------|------|
| `2019-Oct.csv` | 2019-10 |
| `2019-Nov.csv` | 2019-11 |
| `2019-Dec.csv` | 2019-12 |
| `2020-Jan.csv` | 2020-01 |
| `2020-Feb.csv` | 2020-02 |

In [2]:
FILES = ['2019-Oct.csv', '2019-Nov.csv', '2019-Dec.csv', '2020-Jan.csv', '2020-Feb.csv']

file_info = pd.DataFrame({
    '파일': FILES,
    '크기_MB': [round(os.path.getsize(f'{DATA_DIR}/{f}') / 1024**2, 1) for f in FILES],
})
file_info

,파일,크기_MB
0,2019-Oct.csv,460.2
1,2019-Nov.csv,520.6
2,2019-Dec.csv,396.1
3,2020-Jan.csv,478.5
4,2020-Feb.csv,466.2


파일당 400만 행 규모라 전체를 메모리에 올리지 않고, 스키마·자료형만 첫 파일 **샘플**로 확인한다. 행수·이벤트 분포·결측·가격 이상값 등 실제 수치는 적재 이후 3. 검증에서 DB 집계로 확인한다.

In [3]:
# 스키마 확인용 샘플 (첫 파일 상위 20만 행)
sample = pd.read_csv(f'{DATA_DIR}/{FILES[0]}', nrows=200_000)
sample.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,cart,5773203,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
1,2019-10-01 00:00:03 UTC,cart,5773353,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
2,2019-10-01 00:00:07 UTC,cart,5881589,2151191071051219817,NaN,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
3,2019-10-01 00:00:07 UTC,cart,5723490,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
4,2019-10-01 00:00:15 UTC,cart,5881449,1487580013522845895,NaN,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9


In [4]:
sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     200000 non-null  object 
 1   event_type     200000 non-null  object 
 2   product_id     200000 non-null  int64  
 3   category_id    200000 non-null  int64  
 4   category_code  2862 non-null    object 
 5   brand          120806 non-null  object 
 6   price          200000 non-null  float64
 7   user_id        200000 non-null  int64  
 8   user_session   199991 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 13.7+ MB


### 공유 스키마 (9컬럼)
- **event_time**: 이벤트 발생 시각 (UTC)
- **event_type**: 이벤트 유형 (`view` / `cart` / `remove_from_cart` / `purchase`)
- **product_id**: 상품 고유 식별자
- **category_id**: 카테고리 고유 식별자 (숫자)
- **category_code**: 카테고리 코드 문자열
- **brand**: 브랜드명
- **price**: 이벤트 시점 가격
- **user_id**: 사용자 고유 식별자
- **user_session**: 세션 식별자 UUID

---
## 2. 적재 — MySQL 저장

원본 9컬럼을 그대로 보존하는 `events` 테이블 하나에 적재한다. `price`는 0 이하·이상치를 필터하지 않고 원본대로 싣고, 결측이 있는 `category_code`·`brand`·`user_session`은 `NULL`을 허용한다. 인덱스는 적재 속도를 위해 적재 이후 생성한다.

| 테이블 | 내용 |
|--------|------|
| `events` | 이벤트 로그 원본 (5개 월별 CSV 통합, 스키마 보존) |

In [5]:
# local_infile=1: LOAD DATA LOCAL INFILE 벌크 적재를 위한 클라이언트측 활성화
# host=localhost는 PyMySQL이 유닉스 소켓으로 접속(포트 무관)
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{quote_plus(os.getenv('DB_PASSWORD'))}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4",
    connect_args={"local_infile": 1},
)

print(f"{os.getenv('DB_HOST')} / {os.getenv('DB_NAME')}")

localhost / Cosmetics_Funnel


In [6]:
# events 테이블 생성 (재실행 대비 초기화)
CREATE_EVENTS = """
CREATE TABLE events (
    event_time    DATETIME     NOT NULL,
    event_type    VARCHAR(20)  NOT NULL,
    product_id    BIGINT       NOT NULL,
    category_id   BIGINT,
    category_code VARCHAR(120),
    brand         VARCHAR(60),
    price         DECIMAL(10, 2),
    user_id       BIGINT       NOT NULL,
    user_session  CHAR(36)
)
"""

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS events"))
    conn.execute(text(CREATE_EVENTS))

### 월별 적재 (LOAD DATA LOCAL INFILE)
2천만 행 규모라 행 단위 INSERT 대신 DB 벌크 로드로 적재한다(ELT). 파일당 한 번의 `LOAD DATA LOCAL INFILE`로 싣고, 변환은 SQL `SET`절에서만 처리한다.

- `event_time`: ` UTC` 접미사 제거 후 `STR_TO_DATE`로 파싱(UTC 값 유지, 시간대 변환 없음)
- `category_code`·`brand`·`user_session`: 빈 문자열을 `NULLIF`로 `NULL` 처리(원본 결측 보존)
- 그 외 컬럼은 원본 그대로 적재(필터·중복 제거 없음). 인덱스는 로드 후 생성하는 편이 빠르므로 다음 단계로 미룬다.
- 로드마다 경고 건수를 확인해 파싱 실패를 감지한다.

In [7]:
import time

# 원본 CSV → events 벌크 적재. @변수로 받아 SET절에서 변환.
LOAD_SQL = """
LOAD DATA LOCAL INFILE :fpath
INTO TABLE events
FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(@event_time, event_type, product_id, category_id, @category_code, @brand, price, user_id, @user_session)
SET
    event_time = STR_TO_DATE(REPLACE(@event_time, ' UTC', ''), '%Y-%m-%d %H:%i:%s'),
    category_code = NULLIF(@category_code, ''),
    brand = NULLIF(@brand, ''),
    user_session = NULLIF(@user_session, '')
"""

load_log = []
for fname in FILES:
    fpath = os.path.abspath(f'{DATA_DIR}/{fname}')
    t0 = time.time()
    with engine.begin() as conn:
        result = conn.execute(text(LOAD_SQL), {"fpath": fpath})
        warnings = conn.execute(text("SHOW WARNINGS")).fetchall()
    load_log.append({'파일': fname, '적재행수': result.rowcount, '경고': len(warnings), '소요초': round(time.time() - t0)})
    print(f"{fname}: {result.rowcount:,}행 적재 (경고 {len(warnings)}건, {load_log[-1]['소요초']}s)")

pd.DataFrame(load_log)

2019-Oct.csv: 4,102,283행 적재 (경고 0건, 14s)
2019-Nov.csv: 4,635,837행 적재 (경고 0건, 15s)
2019-Dec.csv: 3,533,286행 적재 (경고 0건, 12s)
2020-Jan.csv: 4,264,752행 적재 (경고 0건, 15s)
2020-Feb.csv: 4,156,682행 적재 (경고 0건, 15s)


,파일,적재행수,경고,소요초
0,2019-Oct.csv,4102283,0,14
1,2019-Nov.csv,4635837,0,15
2,2019-Dec.csv,3533286,0,12
3,2020-Jan.csv,4264752,0,15
4,2020-Feb.csv,4156682,0,15


### 인덱스 생성

인덱스 정의는 `sql/indexes.sql` 단일 원천으로 관리한다. 아래 셀은 그 파일을 읽어 `-- name:` 블록별로 실행한다. 적재 후 생성하는 편이 빠르다.

In [8]:
# 인덱스 정의는 sql/indexes.sql 단일 원천 — 파일을 읽어 -- name: 블록별로 실행
import re
from pathlib import Path

def _find_sql(name):
    for base in (Path.cwd(), Path.cwd().parent):
        cand = base / 'sql' / name
        if cand.exists():
            return cand
    raise FileNotFoundError(f'sql/{name} 없음 (cwd={Path.cwd()})')

def _load_queries(path):
    body = Path(path).read_text(encoding='utf-8')
    parts = re.split(r'(?m)^--\s*name:\s*(\w+).*$', body)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

index_queries = _load_queries(_find_sql('indexes.sql'))

with engine.begin() as conn:
    for name, stmt in index_queries.items():
        conn.execute(text(stmt))
        print(f"인덱스 적용: {name}")

---
## 3. 적재 검증

원본 사전 스캔값과 대조해 누락·중복 없이 적재됐는지 확인한다. 사전 스캔 기준값: 총 20,692,840행 / `view` 9,657,821 · `cart` 5,768,333 · `remove_from_cart` 3,979,679 · `purchase` 1,287,007.

In [9]:
EXPECTED_TOTAL = 20_692_840  # 원본 5개 파일 사전 스캔 합계

total = pd.read_sql("SELECT COUNT(*) AS n FROM events", engine).iloc[0, 0]
print(f"적재 행수: {total:,}")
print("행수 일치" if total == EXPECTED_TOTAL else f"불일치: 기대 {EXPECTED_TOTAL:,}")

적재 행수: 20,692,840
행수 일치


In [10]:
monthly = pd.read_sql(
    """
    SELECT
        CONCAT(YEAR(event_time), '-', LPAD(MONTH(event_time), 2, '0')) AS 월,
        COUNT(*) AS 행수
    FROM events
    GROUP BY 월
    ORDER BY 월
    """,
    engine,
)
monthly

,월,행수
0,2019-10,4102283
1,2019-11,4635837
2,2019-12,3533286
3,2020-01,4264752
4,2020-02,4156682


In [11]:
event_dist = pd.read_sql(
    """
    SELECT
        event_type,
        COUNT(*) AS 건수
    FROM events
    GROUP BY event_type
    ORDER BY 건수 DESC
    """,
    engine,
)
event_dist

,event_type,건수
0,view,9657821
1,cart,5768333
2,remove_from_cart,3979679
3,purchase,1287007


In [12]:
uniques = pd.read_sql(
    """
    SELECT
        COUNT(DISTINCT user_id) AS 고유_사용자,
        COUNT(DISTINCT user_session) AS 고유_세션
    FROM events
    """,
    engine,
)
uniques

,고유_사용자,고유_세션
0,1639358,4535941


In [13]:
quality = pd.read_sql(
    """
    SELECT
        SUM(price <= 0) AS 가격_0이하,
        SUM(category_code IS NULL) AS cat_code_결측,
        SUM(brand IS NULL) AS brand_결측,
        SUM(user_session IS NULL) AS session_결측,
        MIN(event_time) AS 최소시각,
        MAX(event_time) AS 최대시각
    FROM events
    """,
    engine,
)
quality

,가격_0이하,cat_code_결측,brand_결측,session_결측,최소시각,최대시각
0,104288.0,20339246.0,8757117.0,4598.0,2019-10-01,2020-02-29 23:59:59


> ### 적재 확인 항목
>
> - 총 행수가 사전 스캔값(20,692,840)과 일치하는지 확인한다.
> - 월별 행수·이벤트 분포가 원본과 일치하는지 확인한다.
> - `가격_0이하`·결측 프로파일은 `01_eda`의 품질 진단·제외 기준 입력으로 인계한다.